# ETL (Extract, Transform, Load) SJW Daily & Weekly Incentive for Riders Apps

>ETL Code to process daily and weekly incentive to used on rider apps

>Created by: lahiardhan@jiwagroup.com

>Created at: 2025-08-06

>Updated at: 2026-11-19

>LATEST UPDATE: 2026-11-19 : kredensial dibuang (config repo + .env), output dibersihkan, diport ke DAG `ksj_rider_commission_production`


# IMPORT LIBRARY

In [ ]:
import datetime
import mysql.connector
from mysql.connector import Error
from mysql.connector import errorcode
import time

from pytz import timezone

import pandas as pd
import numpy as np

import pymysql

import importlib, importlib.util

import warnings
warnings.filterwarnings("ignore")

from sqlalchemy import create_engine, text
import psycopg2

import requests
import json

pd.set_option('display.max_columns', None)

# CONFIG DB

In [ ]:
# Import Config Data
import importlib, importlib.util

def module_from_file(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

config_dw_target = module_from_file("config_dw", "../../../config/config_db_datawarehouse.py")
# config_jiwaplus = module_from_file("config_dw", "../../db_config/config_db_jiwaplus_v2.py")  # tidak dipakai
# config_ksj_link = module_from_file("config_dw", "../../db_config/config_db_ksj_rider_app.py")
config_ksj_link = module_from_file("config_ksj_link", "../../../config/config_db_ksj_link.py")

# FUNCTION

In [ ]:
def convert_data_to_dataframe(data):
    df_all = pd.DataFrame(data)
    for column in ['transactions']:
        df_all[column] = df_all[column].apply(lambda x: x[0] if x else None)  
        if df_all[column][0] is not None: 
            if isinstance(df_all[column][0], dict): 
                for key in df_all[column][0].keys():
                    new_column = f"{column}_{key}"
                    df_all[new_column] = df_all[column].apply(lambda x: x[key] if x else None)
    return df_all

In [ ]:
def expand_nested_lists(df, column_names):
    expanded_data = []
    for index, row in df.iterrows():
        for column_name in column_names:
            data_list = row[column_name]
            if isinstance(data_list, list): 
                for item in data_list:
                    if item is not None:  
                        new_row = row.copy() 
                        for key, value in item.items():
                            new_column_name = f"{column_name}_{key}"  
                            new_row[new_column_name] = value
                        expanded_data.append(new_row)
            else:
                expanded_data.append(row)
    df_expanded = pd.DataFrame(expanded_data)
    return df_expanded

In [ ]:
def adjust_timezone(date_string, hours_offset):
    datetime_obj = datetime.datetime.strptime(date_string, "%Y-%m-%d %H:%M:%S")
    adjusted_datetime = datetime_obj + datetime.timedelta(hours=hours_offset)
    return adjusted_datetime.strftime("%Y-%m-%d %H:%M:%S")

## Riders & Transactions

In [ ]:
def get_riders_mapping(rider_id_list):
    connection = mysql.connector.connect(host=config_dw_target.hostname,
                                 port='3306',
                                 user=config_dw_target.username,
                                 password=config_dw_target.password,
                                 db=config_dw_target.db_resource)
    cursor = connection.cursor()
    
    query_1 = f"""
        SELECT
            DISTINCT
                rider_id,
                rider_name,
                srhm.jilid,
                srhm.hub_name,
                omd.sejuta_jiwa_type,
                om.provinsi,
                om.region,
                om.kota
        FROM sejutajiwa_riders_hubs_mapping srhm
        JOIN outlet_mapping_directory omd
            ON srhm.jilid = omd.jilid
        JOIN outlet_mapping om
            ON srhm.jilid = om.jilid
        WHERE {" OR ".join([f"rider_id LIKE %s" for _ in rider_id_list])};
    """
    
    cursor.execute(query_1, rider_id_list)
    record = cursor.fetchall()

    # Ambil nama kolom
    columns = [col[0] for col in cursor.description]

    # Convert ke DataFrame
    df = pd.DataFrame(record, columns=columns)

    cursor.close()
    connection.close()
    
    return df

In [ ]:
def get_rider_daily_transaction_summaries(start_date_str, end_date_str):
    conn = psycopg2.connect(
        database=config_ksj_link.d_transaction,
        user=config_ksj_link.username,
        password=config_ksj_link.password,
        host=config_ksj_link.hostname,
        port='5432'
    )
    
    query = """
        WITH rider_status AS (
            SELECT
                rider_daily_transaction_summaries.rider_id AS rider_id,
                MAX(rider_daily_transaction_summaries.ownership_status) AS ownership_status
            FROM rider_daily_transaction_summaries
            GROUP BY rider_daily_transaction_summaries.rider_id
        )
        SELECT rider_daily_transaction_summaries.*
        FROM rider_daily_transaction_summaries
        JOIN rider_status ON rider_daily_transaction_summaries.rider_id = rider_status.rider_id
        WHERE rider_daily_transaction_summaries.transaction_date BETWEEN %s AND %s
            AND rider_status.ownership_status = 'KBN'
    """
    
    df = pd.read_sql(query, conn, params=(start_date_str, end_date_str))
    
    conn.close()
    
    return df

In [ ]:
def get_rider_daily_transaction_summary_products(start_date_str, end_date_str):
    conn = psycopg2.connect(
        database=config_ksj_link.d_transaction,
        user=config_ksj_link.username,
        password=config_ksj_link.password,
        host=config_ksj_link.hostname,
        port='5432'
    )
    
    query = """
        WITH rider_status AS (
            SELECT
                rider_daily_transaction_summaries.rider_id AS rider_id,
                MAX(rider_daily_transaction_summaries.ownership_status) AS ownership_status
            FROM rider_daily_transaction_summaries
            GROUP BY rider_daily_transaction_summaries.rider_id
        )
        SELECT rider_daily_transaction_summary_products.*
        FROM rider_daily_transaction_summary_products
        JOIN rider_status
            ON rider_daily_transaction_summary_products.rider_id = rider_status.rider_id
        WHERE rider_daily_transaction_summary_products.transaction_date BETWEEN %s AND %s
            AND rider_status.ownership_status = 'KBN'
    """

    df = pd.read_sql(query, conn, params=(start_date_str, end_date_str))
    
    conn.close()
    
    return df

## Master SKU GP

In [ ]:
def get_master_sku_gp():
    connection = mysql.connector.connect(host=config_dw_target.hostname,
                                 port='3306',
                                 user=config_dw_target.username,
                                 password=config_dw_target.password,
                                 db=config_dw_target.db_resource)
    cursor = connection.cursor()
    
    query_1 = """
        SELECT 
            sku,
            finance_category
        FROM 
            master_sku_gp
        WHERE brand = 'Sejuta Jiwa'
       """
    
    cursor.execute(query_1)
    record = cursor.fetchall()

    cursor.close()
    connection.close()
    
    return record

## Incentive Scheme

In [ ]:
def get_commission_schemes():
    conn = psycopg2.connect(
        database=config_ksj_link.d_transaction,
        user=config_ksj_link.username,
        password=config_ksj_link.password,
        host=config_ksj_link.hostname,
        port='5432'
    )
    
    query = """
        SELECT * FROM commission_schemes
    """
    
    df = pd.read_sql(query, conn)
    
    conn.close()
    
    return df

In [ ]:
def get_commission_scheme_by_weekly_sales():
    conn = psycopg2.connect(
        database=config_ksj_link.d_transaction,
        user=config_ksj_link.username,
        password=config_ksj_link.password,
        host=config_ksj_link.hostname,
        port='5432'
    )
    
    query = """
        SELECT * FROM commission_scheme_by_weekly_sales
    """
    
    df = pd.read_sql(query, conn)
    
    conn.close()
    
    return df

In [ ]:
def get_commission_scheme_by_prod_solds():
    conn = psycopg2.connect(
        database=config_ksj_link.d_transaction,
        user=config_ksj_link.username,
        password=config_ksj_link.password,
        host=config_ksj_link.hostname,
        port='5432'
    )
    
    query = """
        SELECT * FROM commission_scheme_by_prod_solds
    """
    
    df = pd.read_sql(query, conn)
    
    conn.close()
    
    return df

## Extra Incentive Scheme

In [ ]:
def get_commission_extra_schemes():
    conn = psycopg2.connect(
        database=config_ksj_link.d_transaction,
        user=config_ksj_link.username,
        password=config_ksj_link.password,
        host=config_ksj_link.hostname,
        port='5432'
    )
    
    query = """
        SELECT * FROM commission_extra_schemes
    """
    
    df = pd.read_sql(query, conn)
    
    conn.close()
    
    return df

In [ ]:
def get_commission_extra_scheme_by_prod_solds():
    conn = psycopg2.connect(
        database=config_ksj_link.d_transaction,
        user=config_ksj_link.username,
        password=config_ksj_link.password,
        host=config_ksj_link.hostname,
        port='5432'
    )
    
    query = """
        SELECT * FROM commission_extra_scheme_by_prod_solds
    """
    
    df = pd.read_sql(query, conn)
    
    conn.close()
    
    return df

# MAIN PROGRESS

## 1. Get Date

In [ ]:
today = datetime.datetime.strptime("2026-09-14", "%Y-%m-%d").date()
# today = datetime.date.today() - datetime.timedelta(days=1)

start_date = today - datetime.timedelta(days=today.weekday())  # weekday() Senin=0, Minggu=6
end_date = start_date + datetime.timedelta(days=6)

today_str = today.strftime('%Y-%m-%d')
start_date_str = start_date.strftime('%Y-%m-%d')
end_date_str = end_date.strftime('%Y-%m-%d')

print("today:", today_str)
print("start_day:", start_date_str)
print("end_day:", end_date_str)

## 2. Get Riders Mapping

In [ ]:
# df_rider = get_rider_list(start_date, end_date)
# rider_list = df_rider['rider_code'].tolist() 

# df_riders_mapping = get_riders_mapping(rider_list)

# print('Riders Count JIWA+ Prod: ', len(df_rider))
# df_riders_mapping

In [ ]:
# df_rider_list = get_rider_list()

# df_rider_list.head(5)

## 3. Incentive Scheme

In [ ]:
def to_string_cols(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = df[c].apply(
                lambda x: str(int(float(x))) if pd.notna(x) and str(x).replace('.', '', 1).isdigit()
                else 'NaN' if pd.isna(x)
                else str(x).strip()
            )

In [ ]:
df_commission_schemes = get_commission_schemes()

df_commission_scheme_by_weekly_sales = get_commission_scheme_by_weekly_sales()

df_commission_scheme_by_prod_solds = get_commission_scheme_by_prod_solds()

to_string_cols(df_commission_schemes, ['id'])
to_string_cols(df_commission_scheme_by_weekly_sales, ['id', 'commission_scheme_id'])
to_string_cols(df_commission_scheme_by_prod_solds,  ['id', 'commission_scheme_id'])

df_commission_scheme_by_weekly_sales['next_start_range'] = (
    df_commission_scheme_by_weekly_sales.groupby('commission_scheme_id')['start_range'].shift(-1)
)

display(df_commission_schemes.head(3))
display(df_commission_scheme_by_weekly_sales.head(3))
display(df_commission_scheme_by_prod_solds.head(3))

## 4. Extra Incentive Scheme

In [ ]:
df_commission_extra_schemes = get_commission_extra_schemes()
df_commission_extra_scheme_by_prod_solds = get_commission_extra_scheme_by_prod_solds()

to_string_cols(df_commission_extra_schemes, ['id'])
to_string_cols(df_commission_extra_scheme_by_prod_solds,  ['id', 'commission_extra_scheme_id'])

display(df_commission_extra_schemes.head(3))
display(df_commission_extra_scheme_by_prod_solds.head(3))

## 3. Get Transactions Summaries

In [ ]:
df_rider_daily_transaction_summaries = get_rider_daily_transaction_summaries(start_date_str, end_date_str)

to_string_cols(df_rider_daily_transaction_summaries,  ['commission_scheme_id', 'commission_extra_scheme_id'])

df_rider_daily_transaction_summaries.head(4)

## 5. Get Transactions Summary Products

In [ ]:
df_rider_daily_transaction_summary_products = get_rider_daily_transaction_summary_products(start_date_str, end_date_str)

df_rider_daily_transaction_summary_products.head(4)

# Incentive Process

In [ ]:
# --- DTYPE SANITATION ---
# semua *_id & total_operating_days → integer (pakai Int64 biar tetap bisa NA)
int_like_cols = [
    'current_commission_tier',
    # 'commission_scheme_id', 
    # 'commission_extra_scheme_id',
    'rider_id',
    'total_operating_days',
]

## 1. Daily Commission Summary Products

In [ ]:
columns_3 = [
    'rider_id', 'transaction_date', 'product_id', 'sku_code', 'short_code', 'product_name', 'qty', 'product_price',
    'total_price', 'total_bundling_discount', 'total_price_after_discount', 
    
    # 'updated_at', 'created_at',  
    
    'total_product_sold_commission', 'total_product_sold_commission_extra', 
    'commission_per_product_in_idr', 'commission_extra_per_product_in_idr'
]

df_rider_daily_commission_summary_products = pd.DataFrame(columns=columns_3)

df_rider_daily_commission_summary_products

In [ ]:
df_comm_scheme_by_prod_latest = (
    df_commission_scheme_by_prod_solds
    .sort_values("commission_scheme_id", ascending=False)
    .drop_duplicates(subset=["sku_code"], keep="first")
)

In [ ]:
df_daily_trx_products_merged = (
    df_rider_daily_transaction_summary_products
    .fillna(0)
    .groupby(
        # ['rider_code', 'rider_id', 'transaction_date', 'product_id', 'sku_code', 'short_code', 'product_name', 'product_price'],
        ['rider_id', 'transaction_date', 'product_id', 'sku_code', 'short_code', 'product_name', 'product_price'],
        as_index=False
    )
    .agg(
        qty=('qty', 'sum'),
        total_price=('total_price', 'sum'),
        total_bundling_discount=('total_bundling_discount', 'sum'),
        total_price_after_discount=('total_price_after_discount', 'sum')
    )
)

for c in int_like_cols:
    if c in df_daily_trx_products_merged.columns:
        df_daily_trx_products_merged[c] = pd.to_numeric(df_daily_trx_products_merged[c], errors='coerce').astype('Int64')

df_daily_trx_products_merged = df_daily_trx_products_merged.merge(
    df_rider_daily_transaction_summaries[['rider_id', 'transaction_date', 'commission_scheme_id', 'commission_extra_scheme_id', 'commission_type', 'commission_extra_type']],
    on=['rider_id', 'transaction_date'],
    how='left',
)


for col in ["commission_scheme_id", "commission_extra_scheme_id"]:
    if col in df_daily_trx_products_merged.columns:
        df_daily_trx_products_merged[col] = pd.to_numeric(
            df_daily_trx_products_merged[col], errors="coerce"
        ).astype("Int64")

    if col in df_commission_scheme_by_prod_solds.columns:
        df_commission_scheme_by_prod_solds[col] = pd.to_numeric(
            df_commission_scheme_by_prod_solds[col], errors="coerce"
        ).astype("Int64")

    if col in df_commission_extra_scheme_by_prod_solds.columns:
        df_commission_extra_scheme_by_prod_solds[col] = pd.to_numeric(
            df_commission_extra_scheme_by_prod_solds[col], errors="coerce"
        ).astype("Int64")


df_daily_trx_products_merged = df_daily_trx_products_merged.merge(
    df_commission_scheme_by_prod_solds[['commission_scheme_id', 'sku_code', 'commission_in_idr']],
    # df_comm_scheme_by_prod_latest[['sku_code', 'commission_in_idr']],
    on=['commission_scheme_id', 'sku_code'],
    # on=['sku_code'],
    how='left'
)


df_daily_trx_products_merged = df_daily_trx_products_merged.merge(
    df_commission_extra_scheme_by_prod_solds[['sku_code', 'commission_extra_scheme_id', 'commission_in_idr']],
    on=['commission_extra_scheme_id', 'sku_code'],
    how='left',
    suffixes=('', '_extra')
)



df_daily_trx_products_merged = df_daily_trx_products_merged.rename(columns={
    'commission_in_idr': 'commission_per_product_in_idr',
    'commission_in_idr_extra': 'commission_extra_per_product_in_idr'
})

df_daily_trx_products_merged['commission_extra_per_product_in_idr'] = np.where(
    (df_daily_trx_products_merged['commission_extra_type'] == 'product_sold'),
    df_daily_trx_products_merged['commission_extra_per_product_in_idr'],  # nilai asli
    0 # atau 0 kalau mau default 0
)



df_daily_trx_products_merged['total_product_sold_commission'] = df_daily_trx_products_merged['qty'] * df_daily_trx_products_merged['commission_per_product_in_idr']
df_daily_trx_products_merged['total_product_sold_commission_extra'] = df_daily_trx_products_merged['qty'] * df_daily_trx_products_merged['commission_extra_per_product_in_idr']



df_daily_trx_products_merged

In [ ]:
df_rider_daily_commission_summary_products = pd.concat(
    [df_rider_daily_commission_summary_products, df_daily_trx_products_merged[columns_3]],
    ignore_index=True
)

df_rider_daily_commission_summary_products['transaction_date'] = pd.to_datetime(
    df_rider_daily_commission_summary_products['transaction_date'],
    format="%Y-%m-%d",
    errors="coerce"
)

df_rider_daily_commission_summary_products.head(3)

In [ ]:
df_rider_daily_commission_summary_products['transaction_date'] = pd.to_datetime(
    df_rider_daily_commission_summary_products['transaction_date'],
    format="%Y-%m-%d",
    errors="coerce"
)

df_rider_daily_commission_summary_products[
    df_rider_daily_commission_summary_products['transaction_date'].dt.date == datetime.date(2025, 9, 29)
]

### Rekap for One Date and Full Week Date by Product Solds

In [ ]:
# for one day
df_rider_commission_summary_products_one_day = df_rider_daily_commission_summary_products[df_rider_daily_commission_summary_products['transaction_date'].dt.date == today]

# for one week
df_rider_commission_summary_products_one_week = df_rider_daily_commission_summary_products.copy()


grouped_daily_products_scheme = (
    df_rider_commission_summary_products_one_day
    .groupby(['rider_id', 'transaction_date'], as_index=False)
    .agg(
        daily_commission=('total_product_sold_commission', 'sum'),
        daily_commission_extra=('total_product_sold_commission_extra', 'sum')
    )
).reset_index(drop=True)

grouped_weekly_products_scheme = (
    df_rider_commission_summary_products_one_week
    .groupby(['rider_id', 'transaction_date'], as_index=False)
    .agg(
        daily_commission=('total_product_sold_commission', 'sum'),
        daily_commission_extra=('total_product_sold_commission_extra', 'sum')
    )
).reset_index(drop=True)

# ONE DAY ONLY
display(grouped_daily_products_scheme.tail(6))

# FULL ONE WEEK
display(grouped_weekly_products_scheme.tail(6))

In [ ]:
print(df_rider_daily_commission_summary_products['total_price'].sum())
print(df_rider_daily_commission_summary_products['qty'].sum())
# print(len(df_rider_daily_commission_summary_products['rider_id'].unique().tolist()))

## 2. Daily Commission Summaries

In [ ]:
columns_2 = [
    'rider_id', 'transaction_date', 'rider_code', 'hub_code', 'total_price_cash', 'total_price_after_discount_cash', 
    'total_order_cash', 'total_price_qris', 'total_price_after_discount_qris', 'total_order_qris', 'total_item_qty', 
    'commission_scheme_id', 'commission_extra_scheme_id', 'commission_type', 'commission_extra_type', 
    'total_price_jiwaplus', 'total_price_after_discount_jiwaplus', 'total_order_jiwaplus',
    
    'attendance_commission', 'attendance_commission_extra', 
    'daily_commission', 'daily_commission_extra', 
    'daily_sales_commission_percentage', 'daily_sales_commission_extra_percentage', 
]

df_rider_daily_commission_summaries = pd.DataFrame(columns=columns_2)

df_rider_daily_commission_summaries

In [ ]:
df_rider_daily_transaction_summaries

In [ ]:
df_daily_trx_merged = df_rider_daily_transaction_summaries.copy()

for c in int_like_cols:
    if c in df_daily_trx_merged.columns:
        df_daily_trx_merged[c] = pd.to_numeric(df_daily_trx_merged[c], errors='coerce').astype('Int64')

df_daily_trx_merged = df_daily_trx_merged.merge(
    df_commission_schemes[['id', 'type', 'attendance_commission_in_idr', 'daily_commission_in_percentage', 'is_active']],
    left_on='commission_scheme_id',
    right_on='id',
    how='left',
    suffixes=('', '_scheme')
)

df_daily_trx_merged = df_daily_trx_merged.merge(
    df_commission_extra_schemes[['id', 'type', 'attendance_commission_in_idr', 'daily_commission_in_percentage', 'is_active']],
    left_on='commission_extra_scheme_id',
    right_on='id',
    how='left',
    suffixes=('', '_extra')
)

df_daily_trx_merged

In [ ]:
df_daily_trx_merged = df_rider_daily_transaction_summaries.copy()

for c in int_like_cols:
    if c in df_daily_trx_merged.columns:
        df_daily_trx_merged[c] = pd.to_numeric(df_daily_trx_merged[c], errors='coerce').astype('Int64')

df_daily_trx_merged = df_daily_trx_merged.merge(
    df_commission_schemes[['id', 'type', 'attendance_commission_in_idr', 'daily_commission_in_percentage', 'is_active']],
    left_on='commission_scheme_id',
    right_on='id',
    how='left',
    suffixes=('', '_scheme')
)

df_daily_trx_merged = df_daily_trx_merged.merge(
    df_commission_extra_schemes[['id', 'type', 'attendance_commission_in_idr', 'daily_commission_in_percentage', 'is_active']],
    left_on='commission_extra_scheme_id',
    right_on='id',
    how='left',
    suffixes=('', '_extra')
)

df_daily_trx_merged = df_daily_trx_merged.rename(columns={
    'attendance_commission_in_idr': 'attendance_commission',
    'attendance_commission_in_idr_extra': 'attendance_commission_extra',
    'daily_commission_in_percentage': 'daily_sales_commission_percentage',
    'daily_commission_in_percentage_extra': 'daily_sales_commission_extra_percentage',
})

df_daily_trx_merged['total_revenue'] = (
    df_daily_trx_merged['total_price_after_discount_cash'].fillna(0) +
    df_daily_trx_merged['total_price_after_discount_qris'].fillna(0) +
    df_daily_trx_merged['total_price_after_discount_jiwaplus'].fillna(0)
)

df_daily_trx_merged['daily_commission'] = df_daily_trx_merged['total_revenue'] * df_daily_trx_merged['daily_sales_commission_percentage'] * 0.01
df_daily_trx_merged['daily_commission_extra'] = df_daily_trx_merged['total_revenue'] * df_daily_trx_merged['daily_sales_commission_extra_percentage'] * 0.01

df_daily_trx_merged = df_daily_trx_merged.drop(columns=['total_revenue'])

df_daily_trx_merged = df_daily_trx_merged.astype(object).where(pd.notnull(df_daily_trx_merged), np.nan)

df_daily_trx_merged['transaction_date'] = pd.to_datetime(
    df_daily_trx_merged['transaction_date'],
    format="%Y-%m-%d",
    errors="coerce"
)


df_daily_trx_merged.tail(4)

In [ ]:
# Merge grouped daily ke dataframe utama
df_daily_trx_merged = df_daily_trx_merged.merge(
    grouped_weekly_products_scheme[['rider_id', 'transaction_date', 'daily_commission', 'daily_commission_extra']],
    on=['rider_id', 'transaction_date'],
    how='left',
    suffixes=('', '_daily')
)

# Overwrite values only when scheme_id = 1
# mask_prod_scheme = df_daily_trx_merged['commission_scheme_id'] == 210
# mask_extra_prod_scheme = df_daily_trx_merged['commission_extra_scheme_id'] == 1
mask_prod_scheme = df_daily_trx_merged['type'] == 'product_sold'
mask_extra_prod_scheme = df_daily_trx_merged['type_extra'] == 'product_sold'

df_daily_trx_merged.loc[mask_prod_scheme, 'daily_commission'] = df_daily_trx_merged.loc[mask_prod_scheme, 'daily_commission_daily']
df_daily_trx_merged.loc[mask_extra_prod_scheme, 'daily_commission_extra'] = df_daily_trx_merged.loc[mask_extra_prod_scheme, 'daily_commission_extra_daily']

# Drop helper cols if you don't need them
df_daily_trx_merged = df_daily_trx_merged.drop(
    columns=['daily_commission_daily', 'daily_commission_extra_daily']
)

df_daily_trx_merged

In [ ]:
df_rider_daily_commission_summaries = pd.concat(
    [df_rider_daily_commission_summaries, df_daily_trx_merged[columns_2]],
    ignore_index=True
)

df_rider_daily_commission_summaries

### Rekap for One Date and Full Week Date Daily Commission

In [ ]:
# for one day
df_rider_daily_commission_summary_one_day = df_rider_daily_commission_summaries[df_rider_daily_commission_summaries['transaction_date'].dt.date == today]

# for one week
df_rider_daily_commission_summary_one_week = df_rider_daily_commission_summaries.copy()

grouped_weekly_scheme_summaries = (
    df_rider_daily_commission_summary_one_week
    .groupby(['rider_id'], as_index=False)
    .agg(
        total_commission=('daily_commission', 'sum'),
        total_attendance_commission=('attendance_commission', 'sum'),
        total_commission_extra=('daily_commission_extra', 'sum'),
        total_attendance_commission_extra=('attendance_commission_extra', 'sum')
    )
).reset_index(drop=True)

display(grouped_weekly_scheme_summaries.head(4))

In [ ]:
display(df_rider_daily_commission_summary_one_day.head(3))
print(df_rider_daily_commission_summary_one_day['transaction_date'].unique())

## 3. Weekly Commission Summaries

In [ ]:
columns_1 = [[
    'rider_id', 'start_date', 'end_date', 'rider_code', 'hub_code', 'total_revenue', 'total_operating_days', 
    'is_weekly_commission', 'commission_scheme_id', 'current_weekly_commission_percentage', 'current_commission_tier',
    'next_revenue_target', 'total_commission', 'total_attendance_commission', 'total_commission_extra',
    'total_attendance_commission_extra'
]]

df_rider_weekly_commission_summaries = pd.DataFrame(columns=columns_1)

df_rider_weekly_commission_summaries

In [ ]:
# --- WEEKLY ROLLUP (sum, last) ---
df_weekly_calc = (
    df_rider_daily_transaction_summaries
    .groupby(['rider_id', 'rider_code', 'hub_code'], as_index=False)
    .agg({
        'total_price_after_discount_cash': 'sum',
        'total_price_after_discount_qris': 'sum',
        'total_price_after_discount_jiwaplus': 'sum',
        'transaction_date': 'nunique',           # total_operating_days
        'commission_scheme_id': 'last',           # ambil skema terakhir di minggu tsb
        'commission_type': 'last'
    })
    .rename(columns={'transaction_date': 'total_operating_days'})
)

for c in int_like_cols:
    if c in df_weekly_calc.columns:
        df_weekly_calc[c] = pd.to_numeric(df_weekly_calc[c], errors='coerce').astype('Int64')

df_weekly_calc['total_revenue'] = (
    df_weekly_calc['total_price_after_discount_cash'].fillna(0) +
    df_weekly_calc['total_price_after_discount_qris'].fillna(0) +
    df_weekly_calc['total_price_after_discount_jiwaplus'].fillna(0)
)
df_weekly_calc = df_weekly_calc.drop(columns=['total_price_after_discount_cash',
                                              'total_price_after_discount_qris',
                                              'total_price_after_discount_jiwaplus'])

# periode minggu (sudah kamu hitung sebelumnya)
df_weekly_calc['start_date'] = start_date
df_weekly_calc['end_date']   = end_date

# --- PREP SKEMA: tambahkan next tier start (untuk next_revenue_target) ---
df_scheme = df_commission_scheme_by_weekly_sales.copy()
df_scheme = df_scheme.sort_values(['commission_scheme_id', 'start_range'])
df_scheme['next_start_range'] = (
    df_scheme.groupby('commission_scheme_id')['start_range'].shift(-1)
)

dfj = df_weekly_calc.merge(df_scheme, on='commission_scheme_id', how='left')
mask = (dfj['total_revenue'] >= dfj['start_range']) & (dfj['total_revenue'] <= dfj['end_range'])

df_tier = (
    dfj[mask]
    .loc[:, ['rider_id','rider_code','hub_code',
             'commission_in_percentage','attendance_commission_in_idr',
             'tier','next_start_range']]
)

In [ ]:
df_tier

In [ ]:
# --- WEEKLY ROLLUP (sum, last) ---
df_weekly_calc = (
    df_rider_daily_transaction_summaries
    .sort_values('transaction_date')
    .groupby(['rider_id', 'rider_code', 'hub_code'], as_index=False)
    .agg({
        'total_price_after_discount_cash': 'sum',
        'total_price_after_discount_qris': 'sum',
        'total_price_after_discount_jiwaplus': 'sum',
        'transaction_date': 'nunique',           # total_operating_days
        'commission_scheme_id': 'last',           # ambil skema terakhir di minggu tsb
        'commission_type': 'last'
    })
    .rename(columns={'transaction_date': 'total_operating_days'})
)

for c in int_like_cols:
    if c in df_weekly_calc.columns:
        df_weekly_calc[c] = pd.to_numeric(df_weekly_calc[c], errors='coerce').astype('Int64')

df_weekly_calc['total_revenue'] = (
    df_weekly_calc['total_price_after_discount_cash'].fillna(0) +
    df_weekly_calc['total_price_after_discount_qris'].fillna(0) +
    df_weekly_calc['total_price_after_discount_jiwaplus'].fillna(0)
)
df_weekly_calc = df_weekly_calc.drop(columns=['total_price_after_discount_cash',
                                              'total_price_after_discount_qris',
                                              'total_price_after_discount_jiwaplus'])

# periode minggu (sudah kamu hitung sebelumnya)
df_weekly_calc['start_date'] = start_date
df_weekly_calc['end_date']   = end_date

# --- PREP SKEMA: tambahkan next tier start (untuk next_revenue_target) ---
df_scheme = df_commission_scheme_by_weekly_sales.copy()
df_scheme = df_scheme.sort_values(['commission_scheme_id', 'start_range'])
df_scheme['next_start_range'] = (
    df_scheme.groupby('commission_scheme_id')['start_range'].shift(-1)
)

# --- RANGE JOIN: cocokkan tier berdasarkan total_revenue ---
dfj = df_weekly_calc.merge(df_scheme, on='commission_scheme_id', how='left')
mask = (dfj['total_revenue'] >= dfj['start_range']) & (dfj['total_revenue'] <= dfj['end_range'])
df_tier = (
    dfj[mask]
    .loc[:, ['rider_id','rider_code','hub_code',
             'commission_in_percentage','attendance_commission_in_idr',
             'tier','next_start_range']]
)

# gabung balik ke weekly_calc (1 baris per rider)
df_weekly_calc = df_weekly_calc.merge(
    df_tier,
    on=['rider_id','rider_code','hub_code'],
    how='left'
)

# --- KALKULASI KOMISI ---
# boolean: weekly commission aktif jika commission_type = weekly_sales
df_weekly_calc['is_weekly_commission'] = (df_weekly_calc['commission_scheme_id'].map(df_commission_schemes.set_index('id')['type']) == 'weekly_sales')
df_weekly_calc['is_weekly_commission'] = df_weekly_calc['is_weekly_commission'].fillna(False).astype(bool)

# current tier & percentage
df_weekly_calc['current_commission_tier'] = df_weekly_calc['tier']
df_weekly_calc['current_weekly_commission_percentage'] = df_weekly_calc['commission_in_percentage']

# target next tier
df_weekly_calc['next_revenue_target'] = df_weekly_calc['next_start_range']

# amounts
# pct = (df_weekly_calc['current_weekly_commission_percentage'].fillna(0) / 100.0)
df_weekly_calc['total_commission'] = (df_weekly_calc['total_revenue'] * df_weekly_calc['current_weekly_commission_percentage'] * 0.01).fillna(0)
df_weekly_calc['total_attendance_commission'] = (
    df_weekly_calc['total_operating_days'].fillna(0) *
    df_weekly_calc['attendance_commission_in_idr'].fillna(0)
)

# --- SUSUN KE URUTAN KOLOM FINAL & APPEND ---
final_cols = columns_1[0]  # karena columns_1 adalah list di dalam list
df_weekly_calc_final = (
    df_weekly_calc
    .rename(columns={
        # pastikan nama kolom match dengan final schema
        # (id skema tier -> current_commission_tier sudah di-set di atas)
    })
    .reindex(columns=final_cols)
)



# --- SUSUN KE URUTAN KOLOM FINAL & APPEND ---
final_cols = columns_1[0]
df_weekly_calc_final = df_weekly_calc.reindex(columns=final_cols)

try:
    df_rider_weekly_commission_summaries = pd.concat(
        [df_rider_weekly_commission_summaries, df_weekly_calc_final],
        ignore_index=True
    ).reindex(columns=final_cols)
    
except:
    df_rider_weekly_commission_summaries = df_weekly_calc_final.reindex(columns=final_cols)

df_rider_weekly_commission_summaries

In [ ]:
# Merge grouped daily ke dataframe utama
df_rider_weekly_commission_summaries = df_rider_weekly_commission_summaries.merge(
    grouped_weekly_scheme_summaries[['rider_id', 'total_commission', 'total_attendance_commission', 'total_commission_extra', 'total_attendance_commission_extra']],
    on=['rider_id'],
    how='left',
    suffixes=('', '_grp')
)

# Overwrite for non-weekly schemes (daily_sales / product_sold): use weekly sum of daily table
scheme_type = df_rider_weekly_commission_summaries['commission_scheme_id'].map(df_commission_schemes.set_index('id')['type'])
mask_prod_scheme = scheme_type.isin(['daily_sales', 'product_sold'])

df_rider_weekly_commission_summaries.loc[mask_prod_scheme, 'total_commission'] = df_rider_weekly_commission_summaries.loc[mask_prod_scheme, 'total_commission_grp']
df_rider_weekly_commission_summaries.loc[mask_prod_scheme, 'total_attendance_commission'] = df_rider_weekly_commission_summaries.loc[mask_prod_scheme, 'total_attendance_commission_grp']
df_rider_weekly_commission_summaries['total_commission_extra'] = df_rider_weekly_commission_summaries['total_commission_extra_grp']
df_rider_weekly_commission_summaries['total_attendance_commission_extra'] = df_rider_weekly_commission_summaries['total_attendance_commission_extra_grp']

# Drop helper cols if you don't need them
df_rider_weekly_commission_summaries = df_rider_weekly_commission_summaries.drop(
    columns=['total_commission_grp', 'total_attendance_commission_grp', 'total_commission_extra_grp', 'total_attendance_commission_extra_grp']
)

# rename column
df_rider_weekly_commission_summaries = df_rider_weekly_commission_summaries.rename(
    columns={"commission_scheme_id": "weekly_commission_scheme_id"}
)

df_rider_weekly_commission_summaries

In [ ]:
display(df_rider_daily_commission_summary_one_day)
display(df_rider_commission_summary_products_one_day)
display(df_rider_weekly_commission_summaries)

In [ ]:
print('total revenue weekly:', df_rider_weekly_commission_summaries['total_revenue'].sum())
print('total revenue daily:', df_rider_daily_commission_summary_one_week['total_price_cash'].sum() + df_rider_daily_commission_summary_one_week['total_price_qris'].sum() + df_rider_daily_commission_summary_one_week['total_price_jiwaplus'].sum())
print('total revenue daily prod:', df_rider_commission_summary_products_one_week['total_price'].sum())
print('sales commission:', df_rider_weekly_commission_summaries['total_commission'].sum())
print('attd. commission:', df_rider_weekly_commission_summaries['total_attendance_commission'].sum())
print('total payout:', df_rider_weekly_commission_summaries['total_commission'].sum() + df_rider_weekly_commission_summaries['total_attendance_commission'].sum())

# Dump to JIWA+ DB

## Engine Config

In [ ]:
# Engine PostgreSQL pakai psycopg2
engine = create_engine(
    "postgresql+psycopg2://{user}:{pw}@{host}:{port}/{db}".format(
        host=config_ksj_link.hostname,
        port=config_ksj_link.port,
        user=config_ksj_link.username,
        pw=config_ksj_link.password,
        db=config_ksj_link.d_transaction
    )
)

## Batch Insert Config

In [ ]:
# Batch insert function
def batch_insert(dataframe, table_name, engine, batch_size=1000):
    with engine.begin() as conn:  # Transaction
        for i in range(0, len(dataframe), batch_size):
            batch = dataframe.iloc[i:i + batch_size]
            batch.to_sql(
                table_name,
                con=conn,
                if_exists="append",
                index=False,
                method="multi"
            )
            print(f"Inserted rows {i + 1} to {i + len(batch)}")

In [ ]:
now_pg = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3] + " +0700"

for df in [
    df_rider_weekly_commission_summaries,
    df_rider_daily_commission_summary_one_day,
    df_rider_commission_summary_products_one_day
]:
    df["created_at"] = now_pg
    df["updated_at"] = now_pg

In [ ]:
import datetime, random

# bikin random menit dan detik biar gak persis jam 03:00
hour = 3
minute = random.randint(0, 59)
second = random.randint(0, 59)
microsecond = random.randint(0, 999) * 1000  # ambil 3 digit ms

# bikin datetime (contohnya tanggal 2025-08-21, bisa kamu ganti ke tanggal lain)
base_date = datetime.date(2025, 8, 21)
random_time = datetime.datetime.combine(
    base_date,
    datetime.time(hour, minute, second, microsecond)
)

# format sesuai PostgreSQL
now_pg = random_time.strftime("%Y-%m-%d %H:%M:%S.%f")[:-3] + " +0700"

for df in [
    df_rider_weekly_commission_summaries,
    df_rider_daily_commission_summary_one_day,
    df_rider_commission_summary_products_one_day
]:
    df["created_at"] = now_pg
    df["updated_at"] = now_pg


## Dump Weekly Commission

In [ ]:
from sqlalchemy import text

def replace_weekly_continue_id(df, table_name, engine, start_date, end_date, batch_size=1000):
    with engine.begin() as conn:
        # Hapus data lama sesuai rentang tanggal
        conn.execute(
            text(f"""
                DELETE FROM {table_name}
                WHERE start_date = :s AND end_date = :e
            """),
            {"s": start_date, "e": end_date}
        )

        # Ambil max(id) setelah delete
        result = conn.execute(text(f"SELECT COALESCE(MAX(id), 0) FROM {table_name}"))
        max_id = result.scalar()

        # Restart sequence dari max_id + 1
        conn.execute(
            text(f"""
                ALTER SEQUENCE {table_name}_id_seq RESTART WITH {max_id + 1}
            """)
        )

        # Insert data baru
        for i in range(0, len(df), batch_size):
            batch = df.iloc[i:i + batch_size]
            batch.to_sql(
                table_name,
                con=conn,
                if_exists="append",
                index=False,
                method="multi"
            )

    print(f"✅ Data minggu {start_date} - {end_date} diganti. Sequence mulai dari {max_id + 1}.")


In [ ]:
replace_weekly_continue_id(
    df_rider_weekly_commission_summaries,
    "rider_weekly_commission_summaries",
    engine,
    start_date,
    end_date
)

# Dump Daily & Products Sold Commission

In [ ]:
def replace_data_continue_id(df, table_name, engine, date_column, start_date, end_date, batch_size=1000):
    """
    Replace data for a date range in a table using DELETE + INSERT pattern.
    Maintains sequence continuity (no gaps in IDs).
    
    Args:
        df: DataFrame to insert
        table_name: Target table name
        engine: SQLAlchemy engine
        date_column: Name of the date column to filter
        start_date: Start date for deletion range
        end_date: End date for deletion range
        batch_size: Number of rows per batch insert
    """
    with engine.begin() as conn:
        # Hapus data lama untuk range tanggal
        conn.execute(
            text(f"""
                DELETE FROM {table_name}
                WHERE {date_column} >= :start AND {date_column} <= :end
            """),
            {"start": start_date, "end": end_date}
        )

        # Ambil max(id) setelah delete
        result = conn.execute(text(f"SELECT COALESCE(MAX(id), 0) FROM {table_name}"))
        max_id = result.scalar()

        # Restart sequence dari max_id + 1
        conn.execute(
            text(f"""
                ALTER SEQUENCE {table_name}_id_seq RESTART WITH {max_id + 1}
            """)
        )

        # Insert data baru
        for i in range(0, len(df), batch_size):
            batch = df.iloc[i:i + batch_size]
            batch.to_sql(
                table_name,
                con=conn,
                if_exists="append",
                index=False,
                method="multi"
            )

    print(f"✅ Data {table_name} untuk {start_date} - {end_date} diganti. Sequence mulai dari {max_id + 1}.")


In [ ]:
def reset_sequence(engine, table_name, id_col="id", schema="public"):
    with engine.begin() as conn:
        sql = text(f"""
            SELECT setval(
                pg_get_serial_sequence('{schema}.{table_name}', '{id_col}'),
                COALESCE(MAX({id_col}), 0) + 1,
                false
            )
            FROM {schema}.{table_name};
        """)
        conn.execute(sql)


In [ ]:
import pandas as pd

id_cols = ["commission_scheme_id", "commission_extra_scheme_id"]

for col in id_cols:
    if col in df_rider_daily_commission_summary_one_day.columns:
        df_rider_daily_commission_summary_one_day[col] = (
            pd.to_numeric(
                df_rider_daily_commission_summary_one_day[col],
                errors='coerce'  # ubah yang tidak bisa jadi angka → NaN
            ).astype("Int64")    # nullable integer: bisa simpan NaN tanpa error
        )

In [ ]:
import pandas as pd

id_cols = ["commission_scheme_id", "commission_extra_scheme_id"]

for col in id_cols:
    if col in df_rider_daily_commission_summary_one_week.columns:
        df_rider_daily_commission_summary_one_week[col] = (
            pd.to_numeric(
                df_rider_daily_commission_summary_one_week[col],
                errors='coerce'  # ubah yang tidak bisa jadi angka → NaN
            ).astype("Int64")    # nullable integer: bisa simpan NaN tanpa error
        )

In [ ]:
replace_data_continue_id(
    df_rider_daily_commission_summary_one_week,
    "rider_daily_commission_summaries",
    engine,
    date_column="transaction_date",
    start_date=start_date,
    end_date=end_date
)


In [ ]:
replace_data_continue_id(
    df_rider_commission_summary_products_one_week,
    "rider_daily_commission_summary_products",
    engine,
    date_column="transaction_date",
    start_date=start_date,
    end_date=end_date
)


### Send Email Notification

In [ ]:
import os, smtplib, ssl

port = 587  # For starttls
smtp_server = "smtp.gmail.com"
sender_email = "saputra.christabel20@gmail.com"
receiver_email = ["athens.jiwagroup@gmail.com"]
password = os.getenv("GMAIL_SMTP_PASSWORD")

message = """\
Subject: [NEW] - Daily Incentive JIWA+ Rider - DONE

Hi ! This is reporting Data for You !

This message is sent from Jupyter Notebook.

Best Regards,
DatO ( Data autOmation )"""

context = ssl.create_default_context()
with smtplib.SMTP(smtp_server, port) as server:
    server.ehlo()  # Can be omitted
    server.starttls(context=context)
    server.ehlo()  # Can be omitted
    server.login(sender_email, password)
    server.sendmail(sender_email, receiver_email, message)